# Qwen2VL Baseline

Notebook độc lập cho Colab, giữ cùng flow inference với adapter Qwen2VL trong repo.

Chỉ cần sửa cell `CONFIG`, đặc biệt là đường dẫn ảnh/test json và lựa chọn model 2B hay 7B.


In [ ]:
%pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils sentencepiece


In [ ]:
import json
import math
import random
import re
import time
import unicodedata
import warnings
from collections import Counter
from functools import lru_cache
from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

ZERO_WIDTH_RE = re.compile(r"[​‌‍﻿]")
EDGE_PUNCT_RE = re.compile(r'^[\s\.,!?;:\'"“”‘’`~_]+|[\s\.,!?;:\'"“”‘’`~_]+$')
DASH_TRANSLATION = str.maketrans({"–": "-", "—": "-", "−": "-", "‐": "-"})
SIMPLE_NUMBER_WORDS = {
    "không": "0",
    "một": "1",
    "hai": "2",
    "ba": "3",
    "bốn": "4",
    "tư": "4",
    "năm": "5",
    "lăm": "5",
    "sáu": "6",
    "bảy": "7",
    "bẩy": "7",
    "tám": "8",
    "chín": "9",
    "mười": "10",
    "zero": "0",
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
    "ten": "10",
}


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def pick_dtype(device: str, prefer_bfloat16: bool = True) -> torch.dtype:
    if not str(device).startswith("cuda"):
        return torch.float32
    if prefer_bfloat16 and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def build_4bit_config(load_in_4bit: bool, device: str, compute_dtype: torch.dtype):
    if not load_in_4bit:
        return None
    if not str(device).startswith("cuda"):
        raise ValueError("load_in_4bit=True requires CUDA.")
    from transformers import BitsAndBytesConfig

    if compute_dtype not in {torch.float16, torch.bfloat16}:
        compute_dtype = torch.float16
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )


def move_to_device(obj: Any, device: str):
    if isinstance(obj, torch.Tensor):
        return obj.to(device)
    if isinstance(obj, dict):
        return {k: move_to_device(v, device) for k, v in obj.items()}
    if isinstance(obj, list):
        return [move_to_device(v, device) for v in obj]
    if isinstance(obj, tuple):
        return tuple(move_to_device(v, device) for v in obj)
    return obj


def _canonicalize_number(text: str) -> str:
    compact = text.replace(" ", "")
    if re.fullmatch(r"\d{1,3}([.,]\d{3})+", compact):
        return re.sub(r"[.,]", "", compact)
    if re.fullmatch(r"\d+,\d+", compact) and not re.fullmatch(r"\d{1,3}(,\d{3})+", compact):
        return compact.replace(",", ".")
    return text


def normalize_answer(text: str) -> str:
    if text is None:
        return ""

    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.translate(DASH_TRANSLATION)
    text = ZERO_WIDTH_RE.sub("", text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = EDGE_PUNCT_RE.sub("", text)
    text = re.sub(r"[^\w\sÀ-ɏḀ-ỿ\-./,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if text in SIMPLE_NUMBER_WORDS:
        text = SIMPLE_NUMBER_WORDS[text]

    text = _canonicalize_number(text)
    text = re.sub(r"^[\.,!?;:]+|[\.,!?;:]+$", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def exact_match_score(prediction: str, ground_truths: list[str]) -> float:
    pred = normalize_answer(prediction)
    return float(any(normalize_answer(gt) == pred for gt in ground_truths))


def _token_f1_single(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()

    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gt_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def f1_score(prediction: str, ground_truths: list[str]) -> float:
    return max((_token_f1_single(prediction, gt) for gt in ground_truths), default=0.0)


def compute_metrics(predictions: list[str], ground_truths: list[list[str]]) -> dict[str, Any]:
    assert len(predictions) == len(ground_truths)
    if not predictions:
        return {"exact_match": 0.0, "f1": 0.0, "num_samples": 0}

    total_em = sum(exact_match_score(pred, gts) for pred, gts in zip(predictions, ground_truths))
    total_f1 = sum(f1_score(pred, gts) for pred, gts in zip(predictions, ground_truths))
    n = len(predictions)
    return {
        "exact_match": total_em / n,
        "f1": total_f1 / n,
        "num_samples": n,
    }


def load_annotations(json_path: str | Path) -> list[dict]:
    json_path = Path(json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ["annotations", "data", "items", "samples"]:
            if key in data and isinstance(data[key], list):
                return data[key]
    raise ValueError(f"Unsupported annotation format: {json_path}")


def extract_question(ann: dict) -> str:
    for key in ["question", "query", "text"]:
        if ann.get(key):
            return str(ann[key])
    raise KeyError(f"Cannot find question field in sample: {ann}")


def extract_ground_truths(ann: dict) -> list[str]:
    if isinstance(ann.get("answers"), list):
        return [str(x) for x in ann["answers"]]
    if isinstance(ann.get("all_answers"), list):
        return [str(x) for x in ann["all_answers"]]
    if ann.get("answer") is not None:
        return [str(ann["answer"])]
    if ann.get("label") is not None:
        if isinstance(ann["label"], list):
            return [str(x) for x in ann["label"]]
        return [str(ann["label"])]
    return [""]


def extract_image_identifier(ann: dict) -> str:
    for key in ["image_id", "image", "image_name", "file_name", "filename"]:
        value = ann.get(key)
        if value is not None:
            return str(value)
    raise KeyError(f"Cannot find image identifier field in sample: {ann}")


@lru_cache(maxsize=100_000)
def resolve_image_path(image_identifier: str, image_dir: str | Path, image_exts: tuple[str, ...]) -> Path:
    image_dir = Path(image_dir)
    candidate = Path(image_identifier)

    if candidate.suffix:
        direct = image_dir / candidate.name
        if direct.exists():
            return direct
        if candidate.exists():
            return candidate

    stem = candidate.stem if candidate.suffix else candidate.name
    for ext in image_exts:
        path = image_dir / f"{stem}{ext}"
        if path.exists():
            return path

    matches = sorted(image_dir.glob(f"{stem}.*"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Cannot find image for id '{image_identifier}' in {image_dir}")


def build_prompt_for_log(question: str, prompt_template: str) -> str:
    return prompt_template.format(question=question)


def build_multimodal_user_text(question: str, prompt_template: str) -> str:
    prompt = build_prompt_for_log(question, prompt_template).strip()
    prompt = re.sub(r"^\s*<image>\s*\n?", "", prompt, count=1)
    prompt = re.sub(r"^\s*User:\s*", "", prompt, count=1)
    prompt = re.sub(r"\nAssistant:\s*$", "", prompt)
    return prompt.strip()


def build_internvl_question(question: str, prompt_template: str) -> str:
    prompt = build_multimodal_user_text(question, prompt_template)
    return "<image>\n" + prompt


def detect_checkpoint_mode(checkpoint_path: str | None, checkpoint_mode: str = "auto") -> str | None:
    if not checkpoint_path:
        return None
    if checkpoint_mode in {"peft", "merged"}:
        return checkpoint_mode

    checkpoint = Path(checkpoint_path)
    if checkpoint.is_dir() and (checkpoint / "adapter_config.json").exists():
        return "peft"
    return "merged"


def save_run_outputs(
    stats_path: str | Path,
    predictions_path: str | Path,
    stats: dict[str, Any],
    results: list[dict[str, Any]],
    skipped: list[dict[str, Any]] | None = None,
) -> None:
    stats_path = Path(stats_path)
    predictions_path = Path(predictions_path)
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    predictions_path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        "stats": stats,
        "predictions": results,
        "skipped": skipped or [],
    }
    with open(predictions_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)


def chunked(items: list[Any], batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def build_eval_item(ann: dict, config: dict) -> dict:
    question = extract_question(ann)
    ground_truths = extract_ground_truths(ann)
    image_identifier = extract_image_identifier(ann)
    image_path = resolve_image_path(
        image_identifier,
        config["image_dir"],
        tuple(config["image_extensions"]),
    )
    image = Image.open(image_path).convert("RGB")

    return {
        "annotation": ann,
        "question": question,
        "ground_truths": ground_truths,
        "image_id": image_identifier,
        "image_path": str(image_path),
        "question_id": ann.get("question_id", ann.get("id", image_identifier)),
        "prompt": build_prompt_for_log(question, config["prompt_template"]),
        "image": image,
    }


@torch.no_grad()
def run_inference(adapter, annotations: list[dict], config: dict):
    batch_size = max(1, int(config["batch_size"]))
    total_batches = math.ceil(len(annotations) / batch_size) if annotations else 0

    predictions: list[str] = []
    ground_truths: list[list[str]] = []
    results: list[dict[str, Any]] = []
    skipped: list[dict[str, Any]] = []

    start_time = time.time()
    for ann_batch in tqdm(chunked(annotations, batch_size), total=total_batches, desc="Inference"):
        raw_batch = []
        for ann in ann_batch:
            try:
                raw_batch.append(build_eval_item(ann, config))
            except Exception as exc:
                skipped.append(
                    {
                        "question_id": ann.get("question_id", ann.get("id")),
                        "image_id": ann.get("image_id", ann.get("image")),
                        "error": repr(exc),
                    }
                )

        if not raw_batch:
            continue

        inputs = adapter.process_batch(raw_batch, max_length=config["max_length"], training=False)
        inputs = move_to_device(inputs, adapter.device)
        batch_predictions = adapter.generate(
            inputs,
            max_new_tokens=config["generation"]["max_new_tokens"],
            num_beams=config["generation"].get("num_beams", 1),
            temperature=config["generation"]["temperature"],
            top_p=config["generation"]["top_p"],
            do_sample=config["generation"]["do_sample"],
            repetition_penalty=config["generation"].get("repetition_penalty", 1.0),
        )

        for item, pred in zip(raw_batch, batch_predictions):
            gts = item["ground_truths"]
            em = exact_match_score(pred, gts)
            f1 = f1_score(pred, gts)
            predictions.append(pred)
            ground_truths.append(gts)
            results.append(
                {
                    "question_id": item["question_id"],
                    "image_id": item["image_id"],
                    "image_path": item["image_path"],
                    "question": item["question"],
                    "prompt": item["prompt"],
                    "prediction": pred,
                    "normalized_prediction": normalize_answer(pred),
                    "ground_truths": gts,
                    "normalized_ground_truths": [normalize_answer(gt) for gt in gts],
                    "exact_match": em,
                    "f1": f1,
                }
            )

    metrics = compute_metrics(predictions, ground_truths)
    elapsed = time.time() - start_time
    stats = {
        "dataset_name": config["dataset_name"],
        "test_json": config["test_json"],
        "image_dir": config["image_dir"],
        "model_name": adapter.model_name,
        "checkpoint_path": config["checkpoint_path"],
        "checkpoint_mode": config["checkpoint_mode_resolved"],
        "device": adapter.device,
        "dtype": str(adapter.dtype).replace("torch.", "") if hasattr(adapter, "dtype") else None,
        "prompt_template": config["prompt_template"],
        "generation": config["generation"],
        "metrics": metrics,
        "num_results": len(results),
        "num_skipped": len(skipped),
        "elapsed_seconds": round(elapsed, 2),
        "samples_per_second": round(len(results) / elapsed, 4) if elapsed > 0 and results else 0.0,
    }
    return stats, results, skipped


In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

CONFIG = {
    "dataset_name": "ViOCRVQA",
    "image_dir": str(REPO_ROOT / "data" / "ViOCRVQA" / "data"),
    "test_json": str(REPO_ROOT / "data" / "ViOCRVQA" / "test.json"),
    "image_extensions": [".jpg", ".jpeg", ".png", ".webp", ".bmp"],
    "checkpoint_path": None,
    "checkpoint_mode": "auto",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 1,
    "max_samples": None,
    "max_length": 1024,
    "seed": 42,
    "output_dir": str(REPO_ROOT / "baseline" / "outputs"),
    "prompt_template": """<image>
User: Trả lời câu hỏi VQA tiếng Việt dựa trên nội dung trong ảnh.
Hãy trả lời ngắn gọn chỉ bằng một từ, cụm từ hoặc số, không giải thích thêm.
Câu hỏi: {question}
Assistant:""",
    "generation": {
        "temperature": 0.1,
        "top_p": 0.9,
        "max_new_tokens": 32,
        "do_sample": True,
        "num_beams": 1,
        "repetition_penalty": 1.05,
    },
    "model": {
        "variant": "2B",
        "variants": {
            "2B": "Qwen/Qwen2-VL-2B-Instruct",
            "7B": "Qwen/Qwen2-VL-7B-Instruct",
        },
        "min_pixels": 256,
        "max_pixels": 512,
        "load_in_4bit": False,
        "low_cpu_mem_usage": True,
    },
}
CONFIG["model_name"] = CONFIG["model"]["variants"][CONFIG["model"]["variant"]]
CONFIG["stats_path"] = str(Path(CONFIG["output_dir"]) / "qwen2vl_stats.json")
CONFIG["predictions_path"] = str(Path(CONFIG["output_dir"]) / "qwen2vl_predictions.json")
CONFIG


In [ ]:
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration


class AdapterClass:
    def __init__(self, config: dict):
        self.config = config
        self.device = config["device"]
        self.dtype = pick_dtype(self.device)
        self.model_cfg = config["model"]
        self.prompt_template = config["prompt_template"]
        self.model_name = config["model_name"]
        self.model = None
        self.processor = None
        self.pad_token_id = 0

    def _get_device_map(self):
        return "auto" if str(self.device).startswith("cuda") else None

    def _build_processor(self, model_name: str):
        min_pixels = self.model_cfg.get("min_pixels", 256) * 28 * 28
        max_pixels = self.model_cfg.get("max_pixels", 512) * 28 * 28
        return AutoProcessor.from_pretrained(
            model_name,
            min_pixels=min_pixels,
            max_pixels=max_pixels,
        )

    @staticmethod
    def _pad_1d(sequences: list[torch.Tensor], pad_value: int, max_len: int) -> torch.Tensor:
        padded = []
        for seq in sequences:
            pad_len = max_len - seq.size(0)
            padded.append(torch.cat([seq, torch.full((pad_len,), pad_value, dtype=seq.dtype)], dim=0))
        return torch.stack(padded)

    def load_for_inference(self, config: dict) -> None:
        checkpoint_path = config.get("checkpoint_path")
        checkpoint_mode = config.get("checkpoint_mode_resolved")
        load_source = checkpoint_path if checkpoint_mode == "merged" else self.model_name
        proc_source = checkpoint_path if checkpoint_mode == "merged" else self.model_name

        self.processor = self._build_processor(proc_source)
        self.pad_token_id = self.processor.tokenizer.pad_token_id or 0

        load_kwargs = {
            "device_map": self._get_device_map(),
            "low_cpu_mem_usage": self.model_cfg.get("low_cpu_mem_usage", True),
        }
        if self._get_device_map() is None:
            load_kwargs["torch_dtype"] = torch.float32
        else:
            load_kwargs["torch_dtype"] = self.dtype

        bnb_config = build_4bit_config(
            self.model_cfg.get("load_in_4bit", False),
            self.device,
            self.dtype,
        )
        if bnb_config is not None:
            load_kwargs["quantization_config"] = bnb_config

        base_model = Qwen2VLForConditionalGeneration.from_pretrained(load_source, **load_kwargs)
        if self._get_device_map() is None:
            base_model = base_model.to(self.device)

        if checkpoint_mode == "peft":
            from peft import PeftModel

            base_model = PeftModel.from_pretrained(base_model, checkpoint_path).merge_and_unload()

        self.model = base_model.eval()

    def process_batch(self, items: list[dict], max_length: int = 1024, training: bool = False) -> dict:
        del max_length, training
        all_input_ids = []
        all_masks = []
        all_pixel_values = []
        all_grid_thw = []

        for item in items:
            user_text = build_multimodal_user_text(item["question"], self.prompt_template)
            messages_user = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": item["image"]},
                        {"type": "text", "text": user_text},
                    ],
                }
            ]
            prompt_text = self.processor.apply_chat_template(
                messages_user,
                tokenize=False,
                add_generation_prompt=True,
            )
            image_inputs, _ = process_vision_info(messages_user)
            encoded = self.processor(
                text=[prompt_text],
                images=image_inputs,
                padding=False,
                return_tensors="pt",
            )
            all_input_ids.append(encoded.input_ids[0])
            all_masks.append(encoded.attention_mask[0])
            all_pixel_values.append(encoded.pixel_values)
            all_grid_thw.append(encoded.image_grid_thw)

        max_len = max(seq.size(0) for seq in all_input_ids)
        return {
            "input_ids": self._pad_1d(all_input_ids, self.pad_token_id, max_len),
            "attention_mask": self._pad_1d(all_masks, 0, max_len),
            "pixel_values": torch.cat(all_pixel_values, dim=0),
            "image_grid_thw": torch.cat(all_grid_thw, dim=0),
        }

    def generate(
        self,
        inputs: dict,
        max_new_tokens: int = 32,
        num_beams: int = 1,
        temperature: float = 0.1,
        top_p: float = 0.9,
        do_sample: bool = True,
        repetition_penalty: float = 1.05,
    ) -> list[str]:
        input_len = inputs["input_ids"].shape[1]
        gen_inputs = {k: v for k, v in inputs.items() if k != "labels"}
        generated = self.model.generate(
            **gen_inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=self.pad_token_id,
        )
        return [
            self.processor.tokenizer.decode(seq[input_len:], skip_special_tokens=True).strip()
            for seq in generated
        ]


In [ ]:
set_seed(CONFIG["seed"])
CONFIG["checkpoint_mode_resolved"] = detect_checkpoint_mode(
    CONFIG["checkpoint_path"],
    CONFIG.get("checkpoint_mode", "auto"),
)

annotations = load_annotations(CONFIG["test_json"])
if CONFIG.get("max_samples") is not None:
    annotations = annotations[: int(CONFIG["max_samples"])]

print(f"Repo root     : {REPO_ROOT}")
print(f"Dataset       : {CONFIG['dataset_name']}")
print(f"Test file     : {CONFIG['test_json']}")
print(f"Image dir     : {CONFIG['image_dir']}")
print(f"Annotations   : {len(annotations):,}")
print(f"Model         : {CONFIG['model_name']}")
print(f"Checkpoint    : {CONFIG['checkpoint_path']}")
print(f"Device        : {CONFIG['device']}")
print(f"CheckpointMode: {CONFIG['checkpoint_mode_resolved']}")

adapter = AdapterClass(CONFIG)
adapter.load_for_inference(CONFIG)
stats, results, skipped = run_inference(adapter, annotations, CONFIG)
save_run_outputs(CONFIG["stats_path"], CONFIG["predictions_path"], stats, results, skipped)

print(json.dumps(stats, ensure_ascii=False, indent=2))
print(f"Saved stats       -> {CONFIG['stats_path']}")
print(f"Saved predictions -> {CONFIG['predictions_path']}")
print(f"Preview samples   -> {min(3, len(results))}")
results[:3]
